# U-Net による画像セグメンテーション
## 学習目標
1. 事前学習済みの MobileNetV2 をダウンサンプリング層（エンコーダ）に使用し、カスタム畳み込み層をアップサンプリング層（デコーダ）に使用して、改良型 U-Net モデルを定義する方法を学ぶ
2. U-Net モデルをトレーニングし、出力されたセグメンテーション結果を検査する方法を学ぶ
3. U-Net モデルをコンテナ化し、Vertex AI でカスタムトレーニングジョブを送信する方法を学ぶ

このノートブックでは、画像セグメンテーション用のカスタム U-Net モデルを作成してトレーニングする方法を学びます。

画像セグメンテーションは、画像を複数のセグメント（ピクセルの集合）に分割するコンピュータビジョンのタスクであり、画像の表現をより分析しやすい形に単純化することを目的としています。画像セグメンテーションでは、画像内のすべてのピクセルにクラスラベルが割り当てられます。


## セットアップ


In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [ ]:
import time
from datetime import datetime

import keras
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds

keras.backend.clear_session()

PROJECT = !(gcloud config get-value project)
PROJECT = PROJECT[0]
REGION = "us-central1"
BUCKET = PROJECT
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
OUTPUT_DIR = f"gs://{BUCKET}/pets/models/{TIMESTAMP}"

# silence TF info logs
%env TF_CPP_MIN_LOG_LEVEL=3

#### データの可視化


In [ ]:
def read_and_preprocess(data):
    input_image = keras.layers.Resizing(128, 128)(data["image"])
    input_mask = keras.layers.Resizing(128, 128)(data["segmentation_mask"])

    input_image = keras.ops.cast(input_image, "float32")
    input_mask -= 1  # {1,2,3} to {0,1,2}
    return input_image, input_mask

In [ ]:
AUTO = tf.data.AUTOTUNE

# tfds でデータセットを読み込む
dataset, info = tfds.load(
    "oxford_iiit_pet:3.2.0",
    data_dir="gs://asl-public-data/data/tensorflow_datasets",
    with_info=True,
)

train = dataset["train"].map(read_and_preprocess, num_parallel_calls=AUTO)
test = dataset["test"].map(read_and_preprocess, num_parallel_calls=AUTO)

f, ax = plt.subplots(2, 5, figsize=(16, 5))
for idx, (img, mask) in enumerate(train.take(5)):
    ax[0, idx].imshow(keras.utils.array_to_img(img))
    ax[0, idx].axis("off")
    mask = keras.ops.reshape(mask, [128, 128])
    ax[1, idx].imshow(mask.numpy())
    ax[1, idx].axis("off")

### 改良型 U-Net

ここで使用するモデルは、改良型の [U-Net](https://arxiv.org/pdf/1505.04597v1.pdf) です。U-Net は通常、画像をエンコーディングへとダウンサンプリングするエンコーダと、そのエンコーディングを目的のマスクへとアップサンプリングする対称なデコーダで構成されます。デコーダブロックには、エンコーダブロックをデコーダに直接接続するいくつかのスキップ接続（skip connections）があります。

このノートブックでは、事前学習済みの MobileNetV2 を使用してエンコーディングを作成し、アップサンプリング層（stride=2）のセットを使用して目的のマスクに戻します。その際、アップサンプリングのスキップ層が事前学習済みモデルの対応する重みを使用できるように、適切なサイズの層を取り出します。`block_1_expand_relu` や `block_3_expand_relu` などのレイヤー名は、事前学習済み MobileNetV2 のこれらの出力レイヤーに対応しています。事前学習済み MobileNetV2 の全体構造を確認したい場合は、メモリにロードして `keras.utils.plot_model` や `model.summary()` で分析してみてください。

出力は 3 チャンネルで構成される点に注意してください。これは、各ピクセルに 3 つの候補ラベル（背景、輪郭、内部）があるためです。これは 3 つの候補クラスを持つ多クラス分類問題と考えることができます。

![image](https://storage.googleapis.com/asl-public/public-images/unet.png)
元の研究論文からのアーキテクチャ図


In [ ]:
# 入力画像の形状とチャンネル数を指定
INPUT_SHAPE = [128, 128, 3]
OUTPUT_CHANNELS = 3

#### エンコーダ（ダウンサンプリングスタック）の作成


まず、入力画像をダウンサンプリングするエンコーダを作成します。シンプルにするために、事前学習済みモデル（MobileNetV2）を使用します。ネットワーク最上部の全結合層は含めたくないため、`include_top=False` を設定します。


In [ ]:
# モデルをメモリに読み込む
base_model = keras.applications.MobileNetV2(
    input_shape=INPUT_SHAPE, include_top=False
)

U-Net では、エンコーダの出力を使用してデコーダへのスキップ接続を形成することを思い出してください。名前でこれらのレイヤーにアクセスしてエンコーダの出力を作成し、それを `keras.Model` としてインスタンス化します。

このアーキテクチャのトレーニングを高速化するために、エンコーダの重みを凍結（freeze）してトレーニング時に学習されないようにすることができます。これを行うには、`keras.Model` の `trainable` 属性を `False` に設定します。


In [ ]:
# これらのレイヤーの活性化を使用してスキップ接続を形成
layer_names = [
    "block_1_expand_relu",  # 64x64
    "block_3_expand_relu",  # 32x32
    "block_6_expand_relu",  # 16x16
    "block_13_expand_relu",  # 8x8
    "block_16_project",  # 4x4
]

# 出力
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]

# 特徴量抽出モデルを作成
down_stack = keras.Model(
    inputs=base_model.input,
    outputs=base_model_outputs,
    name="pretrained_mobilenet",
)

down_stack.trainable = False

print(f"Encoder Inputs:\n{down_stack.inputs}")
print("Encoder Outputs:")
for i in range(len(down_stack.outputs)):
    print(down_stack.outputs[i])

ご覧のとおり、エンコーダは入力として高さ/幅 `[128, 128]` の画像を想定しており、`[64, 64]`、`[32, 32]`、`[16, 16]`、`[8, 8]`、`[4, 4]` へとダウンサンプリングしています。


#### デコーダ（アップサンプリングスタック）の作成


まず、アップサンプリング層をシンプルな `keras.Sequential` モデルとして返す関数を作成します。アップサンプリング層は画像サイズを 2 倍にする必要があることを思い出してください。これを行うには、`strides=2` と `padding="same"` を指定した[転置畳み込み層（transposed convolutional layer）](https://keras.io/api/layers/convolution_layers/convolution2d_transpose/)を使用できます。


In [ ]:
def upsample(filters, kernel_size, name, strides=2):
    return keras.Sequential(
        [
            keras.layers.Conv2DTranspose(
                filters, kernel_size, strides=strides, padding="same"
            ),
            keras.layers.BatchNormalization(),
            keras.layers.ReLU(),
        ],
        name=name,
    )

upsample 関数を使用して、次元を元に戻すアップサンプリング層を作成します。


In [ ]:
up_stack = [
    upsample(512, 3, "upsample_4x4_to_8x8"),
    upsample(256, 3, "upsample_8x8_to_16x16"),
    upsample(128, 3, "upsample_16x16_to_32x32"),
    upsample(64, 3, "upsample_32x32_to_64x64"),
]

up_stack

入力をエンコーダに渡し、デコーダを通じてアップサンプリングしながらスキップ接続を作成・確立します。


In [ ]:
inputs = keras.layers.Input(shape=INPUT_SHAPE, name="input_image")

# モデルを通じたダウンサンプリング
skips = down_stack(inputs)
x = skips[-1]
skips = reversed(skips[:-1])

# アップサンプリングとスキップ接続の確立
for idx, (up, skip) in enumerate(zip(up_stack, skips)):
    x = up(x)
    concat = keras.layers.Concatenate(name=f"expand_{idx}")
    x = concat([x, skip])

現在のアップスタックの出力は、高さ/幅が `[64, 64]` です。元の入力形状へとアップサンプリングするための `Conv2DTranspose` である最後のレイヤーを手動で作成します。その後、エンドツーエンドのアーキテクチャを `keras.Model` としてインスタンス化します。


In [ ]:
# これがモデルの最後のレイヤーです
last = keras.layers.Conv2DTranspose(
    OUTPUT_CHANNELS, 3, strides=2, padding="same"
)  # 64x64 -> 128x128

x = last(x)
model = keras.Model(inputs=inputs, outputs=x)

model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

keras.utils.plot_model(model, show_shapes=True, dpi=50)

ご覧のとおり、入力は形状 `[128, 128, 3]` の画像です。この画像は事前学習済み MobileNetV2 を使用してダウンサンプリングされ、その後 `Conv2DTranspose` によって元のサイズへとアップサンプリングされます。また、U-Net アーキテクチャに不可欠なスキップ接続にも注目してください。


#### モデルのローカルでのトレーニング
**注**: GPU がない場合、10 エポックのトレーニングには約 45 分かかることがあります。


In [ ]:
EPOCHS = 10
BATCH_SIZE = 8
STEPS_PER_EPOCH = info.splits["train"].num_examples // BATCH_SIZE
VALIDATION_STEPS = info.splits["test"].num_examples // BATCH_SIZE
MODEL_DIR = "./trained_unet"
train = train.batch(BATCH_SIZE).repeat()
test = test.batch(BATCH_SIZE)

In [ ]:
history = model.fit(
    train,
    validation_data=test,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
)

In [ ]:
pd.DataFrame(history.history)[["val_accuracy", "accuracy"]].plot(
    title="Accuracy"
)
pd.DataFrame(history.history)[["val_loss", "loss"]].plot(title="Loss");

#### 予測の可視化
これらは、トレーニング済みモデルの予測を可視化するためのヘルパー関数です。

`create_mask`: 各ピクセルを最も確率の高い「クラス」（背景、輪郭、内部）に割り当てたマスクを返します。

`display`: 入力画像、画像の正解マスク（ラベル）、モデルからの予測マスクをプロットします。

`show_prediction`: 指定された数の入力画像に対して `display` を呼び出します。


In [ ]:
def create_mask(pred_mask):
    """予測マスクから、ピクセルごとに最も高い確率を持つクラスを返します"""
    pred_mask = keras.ops.argmax(pred_mask, axis=-1)
    pred_mask = keras.ops.expand_dims(pred_mask, axis=-1)
    return pred_mask[0]


def display(display_list):
    """入力画像、正解マスク、予測マスクをプロットします"""
    plt.figure(figsize=(15, 15))
    title = ["Input Image", "True Mask", "Predicted Mask"]

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i + 1)
        plt.title(title[i])
        plt.imshow(keras.utils.array_to_img(display_list[i]))
        plt.axis("off")
    plt.show()


def show_predictions(dataset, num):
    """tf.data.Dataset から指定された件数の予測を表示します"""
    for image, mask in dataset.take(num):
        pred_mask = model.predict(image)
        display([image[0], mask[0], create_mask(pred_mask)])

In [ ]:
show_predictions(test, 5)

#### （省略可）Agent Platform（Vertex AI）でのトレーニング
このセクションでは、トレーニングアプリケーションをコンテナ化し、Agent Platform にカスタムトレーニングジョブを送信します。手順は以下のとおりです:

* トレーニングアプリケーションの Python スクリプトを作成する
* トレーニングスクリプトと依存関係をバンドルする Dockerfile を作成する
* Docker イメージをビルドしてプッシュする
* Agent Platform カスタムトレーニングジョブを作成して実行する


In [ ]:
!mkdir ./pets_trainer

In [ ]:
%%writefile ./pets_trainer/train.py

import tensorflow as tf
import keras
import tensorflow_datasets as tfds
import fire

def read_and_preprocess(data):
    input_image = keras.layers.Resizing(128, 128)(data["image"])
    input_mask = keras.layers.Resizing(128, 128)(data["segmentation_mask"])

    input_image = keras.ops.cast(input_image, "float32")  
    input_mask -= 1  # {1,2,3} to {0,1,2}
    return input_image, input_mask
    
    
def create_train_test_datasets(dataset, batch_size):
    train_ds = dataset['train'].map(read_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    test_ds = dataset['test'].map(read_and_preprocess)
    train_ds = train_ds.cache().shuffle(10 * batch_size).batch(batch_size).repeat()
    train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    
    test_ds = test_ds.batch(batch_size)
    return train_ds, test_ds
    
def upsample(filters, size, name):
    return keras.Sequential([
        keras.layers.Conv2DTranspose(filters, size, strides=2, padding='same'),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU()], name=name)

def build_unet(input_shape, output_channels):
    base_model = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False)
    
    # これらのレイヤーの活性化を使用してスキップ接続を形成
    layer_names = [
        'block_1_expand_relu',   # 64x64
        'block_3_expand_relu',   # 32x32
        'block_6_expand_relu',   # 16x16
        'block_13_expand_relu',  # 8x8
        'block_16_project',      # 4x4
    ]
    base_model_outputs = [base_model.get_layer(name).output for name in layer_names]
    
    # 特徴量抽出モデルを作成
    down_stack = keras.Model(inputs=base_model.input, outputs=base_model_outputs,
                                name='pretrained_mobilenet')

    down_stack.trainable = False
    
    up_stack = [
        upsample(512, 3, 'upsample_4x4_to_8x8'),
        upsample(256, 3, 'upsample_8x8_to_16x16'),
        upsample(128, 3, 'upsample_16x16_to_32x32'),
        upsample(64, 3,  'upsample_32x32_to_64x64')
    ]
    
    inputs = keras.layers.Input(shape=input_shape, name='input_image')
    
    # モデルを通じたダウンサンプリング
    skips = down_stack(inputs)
    x = skips[-1]
    skips = reversed(skips[:-1])
    
    # アップサンプリングとスキップ接続の確立
    for idx, (up, skip) in enumerate(zip(up_stack, skips)):
        x = up(x)
        concat = keras.layers.Concatenate(name='expand_{}'.format(idx))
        x = concat([x, skip])
        
    # これがモデルの最後のレイヤーです
    last = keras.layers.Conv2DTranspose(
        output_channels, 3, strides=2,
        padding='same')  #64x64 -> 128x128
    
    x = last(x)
    return keras.Model(inputs=inputs, outputs=x)
    
    
def train_evaluate(
    batch_size=64,
    tf_dataset='oxford_iiit_pet:3.2.0',
    data_dir='gs://asl-public-data/data/tensorflow_datasets',
    input_shape=[128,128,3],
    output_channels=3,
    epochs=10,
    output_dir=None
):
    
    # tfds からデータセットをダウンロード
    dataset, info = tfds.load(tf_dataset, data_dir=data_dir, with_info=True)
    
    # 訓練用とテスト用データセットを作成
    train_ds, test_ds = create_train_test_datasets(dataset, batch_size)
    
    steps_per_epoch = info.splits['train'].num_examples // batch_size
    validation_steps = info.splits['test'].num_examples // batch_size
    
    model = build_unet(input_shape, output_channels)
    model.compile(
        optimizer='adam',
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])
    
    history = model.fit(
        train_ds, 
        validation_data=test_ds,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps)
    
    if output_dir:
        model.save(output_dir)
        
if __name__ == '__main__':
    fire.Fire(train_evaluate)

#### Dockerfile の作成


In [ ]:
%%writefile ./pets_trainer/Dockerfile
FROM us-docker.pkg.dev/vertex-ai/training/tf-gpu.2-17.py310:latest

# fire をインストール
RUN pip install -U fire tensorflow_datasets

# トレーナーコードを Docker イメージにコピー
WORKDIR /app
COPY train.py .

# トレーナーを呼び出すエントリポイントを設定
ENTRYPOINT ["python", "train.py"]

#### コンテナ化されたトレーニングアプリケーションのビルドとプッシュ


In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "pets_trainer_tf"
TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = (
    f"us-docker.pkg.dev/{PROJECT}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{TAG}"
)
TRAINING_CONTAINER_IMAGE_URI

In [ ]:
!gcloud builds submit --machine-type=e2-highcpu-32 --timeout=15m --tag $TRAINING_CONTAINER_IMAGE_URI pets_trainer

#### Agent Platform 上でのカスタムトレーニングジョブの実行


バケットが存在することを確認し、存在しない場合は作成します


In [ ]:
STAGING_BUCKET = f"gs://{BUCKET}-staging"

%env STAGING_BUCKET={STAGING_BUCKET}
%env REGION={REGION}

In [ ]:
%%bash
exists=$(gcloud storage ls --buckets | grep -w ${STAGING_BUCKET}/)

if [ -n "$exists" ]; then
   echo -e "Bucket gs://${BUCKET} already exists."
    
else
   echo "Creating a new GCS bucket."
   gcloud storage buckets create --location=${REGION} ${STAGING_BUCKET}
   echo -e "\nHere are your current buckets:"
   gcloud storage ls
fi

In [ ]:
# ハイパーパラメータ調整ジョブ（カスタムトレーニングジョブ）の設定に使用する変数を指定
DISPLAY_NAME = "pets-segmentation"
JOB_NAME = f"unet-custom-training-{TIMESTAMP}"
EPOCHS = 50

In [ ]:
from google.cloud import aiplatform

# 必要な仕様を定義
worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-8",
            "accelerator_type": "NVIDIA_TESLA_T4",
            "accelerator_count": 1,
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": TRAINING_CONTAINER_IMAGE_URI,
            "args": [f"--output_dir={OUTPUT_DIR}", f"--epochs={EPOCHS}"],
        },
    }
]

custom_job = aiplatform.CustomJob(
    display_name=JOB_NAME,
    project=PROJECT,
    worker_pool_specs=worker_pool_specs,
    staging_bucket=STAGING_BUCKET,
)

In [ ]:
custom_job.run()

お疲れ様でした！画像セグメンテーション用のカスタム U-Net モデルのトレーニングに成功し、Agent Platform を使用してモデルトレーニングを運用化することができました。
